# Allocate and Free

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import SVG, display
from sympy import Symbol, Max

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.qec_gate_cost import QECGateCost, query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import BookkeepingCall
from flagquantum.core.ir.program.register_types import unsigned_register_type
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_resource_call_graph_dot, render_call_graph_svg

## Allocate

Declare an output-only word with a quantum data type. A clean allocation
uses the caller's clean workspace pool. Set the dirty flag to borrow an
existing word from the separate borrowed pool. Binding chooses physical
wires in the order supplied by the caller.

In [ ]:
from flagquantum.compiler.registers.component_builder import ComponentProgramBuilder
from flagquantum.compiler.registers.register_binding import bind_register_program


def show_terminal(call, name):
    dot = export_resource_call_graph_dot(call, max_depth=1, title=lambda _: name)
    display(SVG(render_call_graph_svg(dot)))
    assert query_qec_gate_cost(call) == QECGateCost()

### Example Instances

In [ ]:
n = Symbol("n")
allocate = BookkeepingCall("allocate", (n, False))
allocate_signature = Signature((
    Register("reg", unsigned_register_type(n), side=RegisterSide.RIGHT),
))

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Allocate", allocate_signature, {})])))

### Call Graph

In [ ]:
show_terminal(allocate, "Allocate")
print("Leaf multiplicity", 1)

## Free

Consume an input-only word and return its wires to the owning pool.
Uncompute a clean word to zero before releasing it. Restore borrowed
wires to their original joint state, including any entanglement with
other wires. Scheduling records lifetime and provenance. The caller
and the operation are responsible for restoring the numerical state.

In the source tensor formulation, Free contracts the word with a zero
bra. The result can have norm below one if the word had nonzero
components. The tensor diagnostic below evaluates this projection
explicitly. An ordinary circuit lifetime release does not perform it.

In [ ]:
from flagquantum.compiler._component_trace.resource_width import query_resource_logical_width
from flagquantum.compiler.registers.layout.int_vector import int_effect_bindings
from flagquantum.simulation.tensor_network.entrypoints import (
    build_tensor_network, tensor_network_basis_effect,
)

### Example Instances

In [ ]:
free = BookkeepingCall("free", (n, False))
free_signature = Signature((
    Register("reg", unsigned_register_type(n), side=RegisterSide.LEFT),
))
assert query_resource_logical_width(allocate) == query_resource_logical_width(free) == Max(0, n)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Free", free_signature, {})])))

### Call Graph

In [ ]:
show_terminal(free, "Free")
print("Leaf multiplicity", 1)

### Bind the lifetime and check the clean state

Allocate, split, join and release a ten-wire word. The circuit is
gate-free and its clean workspace stays at zero. Prepare a separate
superposition to see the unnormalized Free contraction.

In [ ]:
wires = tuple(range(10))
builder = ComponentProgramBuilder(Signature([]))
word = builder.allocate(dtype=unsigned_register_type(len(wires)))
builder.free(builder.join(builder.split(word), word.register.dtype))
bound = bind_register_program(builder.finalize({}), (), clean=wires)
assert [(e.kind, e.wires) for e in bound.events] == [("allocate", wires), ("free", wires)]
assert bound.circuit.instructions == () and bound.outputs == ()

dtype = torch.complex128
circuit = Circuit(len(wires), dtype=dtype)
circuit.append(FieldOperation(len(wires), bound.circuit.instructions))
state = circuit.state().flatten()
assert state[0].item() == 1 and torch.count_nonzero(state[1:]) == 0

superposition = Circuit(len(wires), dtype=dtype)
for wire in wires:
    superposition.h(wire)
plan = build_tensor_network(superposition, dtype=dtype)
projected = tensor_network_basis_effect(plan, int_effect_bindings(wires, 0))
torch.testing.assert_close(projected, torch.full((1, 1), 2**-5, dtype=dtype))